In [ ]:
# In[1]:

import pandas as pd

# Load files
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# Parse timestamps to UTC datetime
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# Helper to build overview and top lists
def build_summary(df, component_col, name_col):
    rows = df.shape[0]
    time_min = df['timestamp'].min()
    time_max = df['timestamp'].max()
    overview = pd.DataFrame([{
        'rows': rows,
        'time_min_utc': time_min,
        'time_max_utc': time_max
    }])
    comp_top = df.groupby(component_col).size().reset_index(name='count').sort_values('count', ascending=False).head(20)
    name_top = df.groupby(name_col).size().reset_index(name='count').sort_values('count', ascending=False).head(20)
    return overview, comp_top, name_top

# Metric summary
metric_overview, metric_cmdb_top, metric_kpi_top = build_summary(metric, 'cmdb_id', 'kpi_name')

# Trace summary
trace_overview, trace_cmdb_top, trace_name_top = build_summary(trace, 'cmdb_id', 'trace_name')

# Log summary
log_overview, log_cmdb_top, log_name_top = build_summary(log, 'cmdb_id', 'log_name')

# Error logs summary (message samples counted)
error_overview, error_cmdb_top, error_msg_top = build_summary(error_logs, 'cmdb_id', 'message')

# Keep outputs compact: show only top-20 tables and single-row overviews
metric_overview, metric_cmdb_top, metric_kpi_top, trace_overview, trace_cmdb_top, trace_name_top, log_overview, log_cmdb_top, log_name_top, error_overview, error_cmdb_top, error_msg_top

```
Out[1]:
```
from datetime import datetime

def fmt_top(df, namecol, limit=20):
    if df is None or df.empty:
        return "none"
    pairs = [f"{row[0]} ({row[1]})" for row in df[[namecol, 'count']].head(limit).itertuples(index=False, name=None)]
    return ", ".join(pairs)

summary = (
    "metric.csv: {rows} rows; time range {tmin} to {tmax} UTC. "
    "Top components (count): {cmps}. "
    "Top KPIs (count): {kpis}."
).format(
    rows=int(metric_overview.loc[0, 'rows']),
    tmin=str(metric_overview.loc[0, 'time_min_utc']),
    tmax=str(metric_overview.loc[0, 'time_max_utc']),
    cmps=fmt_top(metric_cmdb_top, 'cmdb_id'),
    kpis=fmt_top(metric_kpi_top, 'kpi_name')
) + "\n\n" + (
    "trace.csv: {rows} rows; no trace data available."
).format(rows=int(trace_overview.loc[0, 'rows'])) + "\n\n" + (
    "log.csv: {rows} rows; time range {tmin} to {tmax} UTC. "
    "Top components (count): {cmps}. "
    "Top log names (count): {logs}."
).format(
    rows=int(log_overview.loc[0, 'rows']),
    tmin=str(log_overview.loc[0, 'time_min_utc']),
    tmax=str(log_overview.loc[0, 'time_max_utc']),
    cmps=fmt_top(log_cmdb_top, 'cmdb_id'),
    logs=fmt_top(log_name_top, 'log_name')
) + "\n\n" + (
    "error_logs.csv: {rows} rows; no error log messages present."
).format(rows=int(error_overview.loc[0, 'rows']))

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   rows              time_min_utc              time_max_utc
0  1925 2024-01-20 06:12:00+00:00 2024-01-20 06:36:00+00:00,               cmdb_id  count
5              orders    200
4           front-end    175
0               carts    175
2           catalogue    150
13               user    150
12           shipping    150
7             payment    150
6           orders-db    100
3        catalogue-db    100
1            carts-db    100
8        queue-master    100
11         session-db    100
9            rabbitmq    100
14            user-db    100
10  rabbitmq-exporter     75,      kpi_name  count
0         cpu    375
5         mem    375
6      socket    375
1      diskio    225
4  latency-90    175
3  latency-50    175
7    workload    175
2       error     50,    rows time_min_utc time_max_utc
0     0          NaT          NaT, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [trace_name, count]
Index: [],    rows              time_min_utc              time_max_utc
0   458 2024-01-20 06:12:00+00:00 2024-01-20 06:36:00+00:00,          cmdb_id  count
2      catalogue     50
4         orders     50
3      front-end     50
10          user     50
9       shipping     50
6        payment     50
7   queue-master     50
1       carts-db     48
0          carts     48
8     session-db     10
5      orders-db      2,           log_name  count
0  log.error_count    229
1    log.row_count    229,    rows time_min_utc time_max_utc
0     0          NaT          NaT, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [message, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing 'metric' DataFrame loaded earlier; ensure timestamp is datetime UTC
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# 1) Compute global P95 thresholds per (cmdb_id, kpi_name) using the full series (before any window filtering)
thresholds = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='threshold_p95')
)

# 2) Filter metric rows to the incident window (UTC)
start = pd.to_datetime('2024-01-20 06:09:06', utc=True)
end = pd.to_datetime('2024-01-20 06:39:06', utc=True)
mw = metric[(metric['timestamp'] >= start) & (metric['timestamp'] <= end)].copy()

# 3) Merge windowed rows with thresholds so each row knows its group's threshold
mw = mw.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Compute per-row breach flag
mw['above_threshold'] = mw['value'] > mw['threshold_p95']

# 5) Aggregate per (cmdb_id, kpi_name) for rows present in window
agg = (
    mw
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        threshold_p95=('threshold_p95', 'first'),  # same within group
        points_in_window=('value', 'size'),
        points_above_threshold=('above_threshold', 'sum'),
        earliest_breach_time_utc=('timestamp', lambda x: x[mw.loc[x.index, 'above_threshold']].min() if mw.loc[x.index, 'above_threshold'].any() else pd.NaT),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# 6) Calculate ratio_max_to_threshold (handle zero thresholds)
agg['ratio_max_to_threshold'] = np.where(
    agg['threshold_p95'] > 0,
    agg['max_value_in_window'] / agg['threshold_p95'],
    np.nan
)

# 7) Keep only pairs that had at least one data point in the incident window
result = agg[agg['points_in_window'] > 0].copy()

# 8) Sort and limit to top 20 rows as specified
result_sorted = result.sort_values(
    by=['points_above_threshold', 'ratio_max_to_threshold'],
    ascending=[False, False]
).head(20)

# 9) Count of distinct (cmdb_id, kpi_name) pairs analyzed (i.e., present in the window)
pair_count = int(result.shape[0])

# Keep output compact: selected columns in requested order
final_table = result_sorted[
    ['cmdb_id', 'kpi_name', 'threshold_p95', 'points_in_window', 'points_above_threshold',
     'earliest_breach_time_utc', 'max_value_in_window', 'ratio_max_to_threshold']
]

final_table, pair_count

```
Out[2]:
```
summary = (
    "I computed a global P95 threshold per (cmdb_id, kpi_name) using the full series, then examined data "
    "in the incident window 2024-01-20 06:09:06 to 2024-01-20 06:39:06 UTC. "
    "In total 77 distinct (cmdb_id, kpi_name) pairs had data in the window and were analyzed.\n\n"
    "Findings (top examples sorted by breaches then ratio):\n"
    "1) session-db / diskio: threshold_p95 ≈ 16855.27, points_in_window = 25, points_above_threshold = 2, "
    "earliest breach = 2024-01-20 06:20:00 UTC, max_in_window ≈ 36071.16 (ratio ≈ 2.14)\n"
    "2) shipping / latency-90: threshold_p95 ≈ 0.0059649, points_in_window = 25, points_above_threshold = 2, "
    "earliest breach = 2024-01-20 06:12:00 UTC, max_in_window ≈ 0.0073101 (ratio ≈ 1.23)\n"
    "3) shipping / cpu: threshold_p95 ≈ 0.66817, points_in_window = 25, points_above_threshold = 2, "
    "earliest breach = 2024-01-20 06:14:00 UTC, max_in_window ≈ 0.73949 (ratio ≈ 1.11)\n"
    "4) queue-master / cpu: threshold_p95 ≈ 0.59096, points_in_window = 25, points_above_threshold = 2, "
    "earliest breach = 2024-01-20 06:26:00 UTC, max_in_window ≈ 0.63086 (ratio ≈ 1.07)\n"
    "5) payment / cpu: threshold_p95 ≈ 0.09531, points_in_window = 25, points_above_threshold = 2, "
    "earliest breach = 2024-01-20 06:27:00 UTC, max_in_window ≈ 0.10166 (ratio ≈ 1.07)\n\n"
    "Overall pattern: many component/KPI pairs (top examples shown) had 25 samples in the window and exactly 2 "
    "points above their global P95. Most breaches are modest (ratios ~1.02–1.23) except session-db/diskio which "
    "peaked at ~2.14× its P95. "
    "If you want, I can return the full compact table (up to 20 rows) or adjust sorting/filters for deeper inspection."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  threshold_p95  points_in_window  points_above_threshold  earliest_breach_time_utc  max_value_in_window  ratio_max_to_threshold
58         session-db      diskio   1.685527e+04                25                       2 2024-01-20 06:20:00+00:00         3.607116e+04                2.140052
63           shipping  latency-90   5.964862e-03                25                       2 2024-01-20 06:12:00+00:00         7.310139e-03                1.225534
61           shipping         cpu   6.681714e-01                25                       2 2024-01-20 06:14:00+00:00         7.394933e-01                1.106742
62           shipping  latency-50   3.250233e-03                25                       2 2024-01-20 06:12:00+00:00         3.480029e-03                1.070701
46       queue-master         cpu   5.909618e-01                25                       2 2024-01-20 06:26:00+00:00         6.308620e-01                1.067517
40            payment         cpu   9.531025e-02                25                       2 2024-01-20 06:27:00+00:00         1.016626e-01                1.066649
37          orders-db      diskio   8.701988e+05                25                       2 2024-01-20 06:18:00+00:00         9.268470e+05                1.065098
47       queue-master      diskio   5.462565e+04                25                       2 2024-01-20 06:14:00+00:00         5.792126e+04                1.060331
65           shipping      socket   6.320000e+00                25                       2 2024-01-20 06:18:00+00:00         6.700000e+00                1.060127
26          front-end      socket   8.316667e+00                25                       2 2024-01-20 06:30:00+00:00         8.666667e+00                1.042084
0               carts         cpu   1.809197e+00                25                       2 2024-01-20 06:14:00+00:00         1.878871e+00                1.038511
35             orders    workload   2.187767e+00                25                       2 2024-01-20 06:19:00+00:00         2.267000e+00                1.036217
69               user  latency-90   6.445360e-03                25                       2 2024-01-20 06:30:00+00:00         6.671375e-03                1.035066
31             orders  latency-50   4.587663e-02                25                       2 2024-01-20 06:27:00+00:00         4.740596e-02                1.033336
5               carts      socket   1.779667e+01                25                       2 2024-01-20 06:12:00+00:00         1.829630e+01                1.028074
21          front-end         cpu   4.779050e+00                25                       2 2024-01-20 06:31:00+00:00         4.892754e+00                1.023792
74            user-db      diskio   1.575411e+06                25                       2 2024-01-20 06:14:00+00:00         1.609444e+06                1.021603
50           rabbitmq         cpu   8.878228e-01                25                       2 2024-01-20 06:12:00+00:00         9.067351e-01                1.021302
54  rabbitmq-exporter         cpu   1.449896e-04                25                       2 2024-01-20 06:34:00+00:00         1.477709e-04                1.019183
17       catalogue-db         cpu   2.137098e-01                25                       2 2024-01-20 06:15:00+00:00         2.173236e-01                1.016910, 77)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing `metric` DataFrame; ensure timestamp parsed as UTC datetime
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# 1) Compute global P95 thresholds per (cmdb_id, kpi_name) using the full series (before any window filtering)
thresholds = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='threshold_p95')
)

# 2) Filter metric rows to the incident window (UTC)
start = pd.to_datetime('2024-01-20 06:09:06', utc=True)
end = pd.to_datetime('2024-01-20 06:39:06', utc=True)
mw = metric[(metric['timestamp'] >= start) & (metric['timestamp'] <= end)].copy()

# 3) Merge windowed rows with thresholds so each row knows its group's threshold
mw = mw.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Mark anomaly rows: value > threshold_p95 (treat missing threshold as non-anomalous)
mw['anomaly'] = mw['value'] > mw['threshold_p95']

# 5) Keep only anomalous rows for fault grouping
anoms = mw[mw['anomaly']].copy()

# If no anomalies, prepare empty results
if anoms.empty:
    faults_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','threshold_p95','fault_start_utc','fault_end_utc',
        'duration_minutes','points_in_fault','max_value_in_fault','ratio_max_to_threshold'
    ])
    count_ge_1_5 = 0
else:
    # 6) Sort anomalies and compute time gaps within each (cmdb_id,kpi_name)
    anoms = anoms.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)

    # compute time difference in seconds within each group
    anoms['prev_ts'] = anoms.groupby(['cmdb_id','kpi_name'])['timestamp'].shift(1)
    anoms['gap_s'] = (anoms['timestamp'] - anoms['prev_ts']).dt.total_seconds().fillna(0)

    # 7) Start a new fault when gap > 60 seconds
    anoms['new_fault'] = (anoms['gap_s'] > 60).astype(int)

    # 8) Assign fault ids cumulative per (cmdb_id,kpi_name)
    anoms['fault_seq'] = anoms.groupby(['cmdb_id','kpi_name'])['new_fault'].cumsum()

    # 9) Aggregate per fault
    grouped = anoms.groupby(['cmdb_id','kpi_name','fault_seq'], dropna=False)
    faults = grouped.agg(
        threshold_p95=('threshold_p95', 'first'),
        fault_start_utc=('timestamp', 'min'),
        fault_end_utc=('timestamp', 'max'),
        points_in_fault=('timestamp', 'size'),
        max_value_in_fault=('value', 'max')
    ).reset_index(drop=False)

    # 10) Compute duration_minutes as inclusive minute count: floor(diff_minutes)+1
    faults['duration_minutes'] = ((faults['fault_end_utc'] - faults['fault_start_utc']).dt.total_seconds() / 60).astype(int) + 1

    # 11) Compute ratio_max_to_threshold (handle non-positive thresholds)
    faults['ratio_max_to_threshold'] = np.where(
        faults['threshold_p95'] > 0,
        faults['max_value_in_fault'] / faults['threshold_p95'],
        np.nan
    )

    # 12) Select and order columns as requested
    faults_df = faults[[
        'cmdb_id','kpi_name','threshold_p95','fault_start_utc','fault_end_utc',
        'duration_minutes','points_in_fault','max_value_in_fault','ratio_max_to_threshold'
    ]].copy()

    # 13) Sort by ratio descending and limit to top 20
    faults_df = faults_df.sort_values(by='ratio_max_to_threshold', ascending=False).head(20).reset_index(drop=True)

    # 14) Count faults with ratio >= 1.5
    count_ge_1_5 = int((faults_df['ratio_max_to_threshold'] >= 1.5).sum())

# Final outputs (compact)
faults_df, count_ge_1_5

```
Out[3]:
```
summary = (
    "I computed global P95 per (cmdb_id, kpi_name), filtered metric data to 2024-01-20 06:09:06–06:39:06 UTC, "
    "marked rows > P95 as anomalies, and grouped consecutive-minute anomalies into faults.\n\n"
    "Key findings (top 20 faults by ratio):\n"
    "- Total faults returned: 20 (table limited to top 20 by ratio).\n"
    "- Highest-severity fault: session-db / diskio — threshold_p95 ≈ 16855.27; fault at 2024-01-20 06:36 UTC; "
    "duration 1 minute; max ≈ 36071.16; ratio ≈ 2.14.\n"
    "- Only one fault has ratio >= 1.5 (the session-db / diskio fault).\n"
    "- Most faults are short (single-minute breaches); a few span 2 consecutive minutes (e.g., shipping/socket at 06:18–06:19 UTC and front-end/cpu at 06:31–06:32 UTC).\n"
    "- Typical breach magnitudes are modest (ratio mostly ~1.02–1.23) except the session-db/diskio outlier at ~2.14×.\n\n"
    "If you want, I can (a) return the full faults table, (b) show all faults (not just top 20), or (c) expand inspection for the session-db diskio series."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  threshold_p95           fault_start_utc             fault_end_utc  duration_minutes  points_in_fault  max_value_in_fault  ratio_max_to_threshold
0     session-db      diskio   1.685527e+04 2024-01-20 06:36:00+00:00 2024-01-20 06:36:00+00:00                 1                1        3.607116e+04                2.140052
1       shipping  latency-90   5.964862e-03 2024-01-20 06:33:00+00:00 2024-01-20 06:33:00+00:00                 1                1        7.310139e-03                1.225534
2       shipping         cpu   6.681714e-01 2024-01-20 06:14:00+00:00 2024-01-20 06:14:00+00:00                 1                1        7.394933e-01                1.106742
3       shipping  latency-50   3.250233e-03 2024-01-20 06:33:00+00:00 2024-01-20 06:33:00+00:00                 1                1        3.480029e-03                1.070701
4   queue-master         cpu   5.909618e-01 2024-01-20 06:32:00+00:00 2024-01-20 06:32:00+00:00                 1                1        6.308620e-01                1.067517
5        payment         cpu   9.531025e-02 2024-01-20 06:32:00+00:00 2024-01-20 06:32:00+00:00                 1                1        1.016626e-01                1.066649
6      orders-db      diskio   8.701988e+05 2024-01-20 06:32:00+00:00 2024-01-20 06:32:00+00:00                 1                1        9.268470e+05                1.065098
7   queue-master      diskio   5.462565e+04 2024-01-20 06:14:00+00:00 2024-01-20 06:14:00+00:00                 1                1        5.792126e+04                1.060331
8       shipping      socket   6.320000e+00 2024-01-20 06:18:00+00:00 2024-01-20 06:19:00+00:00                 2                2        6.700000e+00                1.060127
9      front-end      socket   8.316667e+00 2024-01-20 06:30:00+00:00 2024-01-20 06:30:00+00:00                 1                1        8.666667e+00                1.042084
10         carts         cpu   1.809197e+00 2024-01-20 06:28:00+00:00 2024-01-20 06:28:00+00:00                 1                1        1.878871e+00                1.038511
11  queue-master      diskio   5.462565e+04 2024-01-20 06:19:00+00:00 2024-01-20 06:19:00+00:00                 1                1        5.670013e+04                1.037976
12        orders    workload   2.187767e+00 2024-01-20 06:36:00+00:00 2024-01-20 06:36:00+00:00                 1                1        2.267000e+00                1.036217
13          user  latency-90   6.445360e-03 2024-01-20 06:35:00+00:00 2024-01-20 06:35:00+00:00                 1                1        6.671375e-03                1.035066
14        orders  latency-50   4.587663e-02 2024-01-20 06:33:00+00:00 2024-01-20 06:33:00+00:00                 1                1        4.740596e-02                1.033336
15  queue-master         cpu   5.909618e-01 2024-01-20 06:26:00+00:00 2024-01-20 06:26:00+00:00                 1                1        6.093321e-01                1.031086
16         carts      socket   1.779667e+01 2024-01-20 06:12:00+00:00 2024-01-20 06:12:00+00:00                 1                1        1.829630e+01                1.028074
17      shipping  latency-90   5.964862e-03 2024-01-20 06:12:00+00:00 2024-01-20 06:12:00+00:00                 1                1        6.107790e-03                1.023962
18     front-end         cpu   4.779050e+00 2024-01-20 06:31:00+00:00 2024-01-20 06:32:00+00:00                 2                2        4.892754e+00                1.023792
19       user-db      diskio   1.575411e+06 2024-01-20 06:34:00+00:00 2024-01-20 06:34:00+00:00                 1                1        1.609444e+06                1.021603, 1)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing 'log' DataFrame; ensure timestamp is parsed to UTC
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# 1) Compute global P95 thresholds per (cmdb_id, log_name) using the full series
log_thresholds = (
    log
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='threshold_p95')
)

# 2) Filter log rows to the incident window and to the specified components
start = pd.to_datetime('2024-01-20 06:09:06', utc=True)
end = pd.to_datetime('2024-01-20 06:39:06', utc=True)
components = ["carts","catalogue","orders","payment","user"]

log_win = log[
    (log['timestamp'] >= start) &
    (log['timestamp'] <= end) &
    (log['cmdb_id'].isin(components))
].copy()

# 3) Merge with thresholds (thresholds computed from full series)
log_win = log_win.merge(log_thresholds, on=['cmdb_id', 'log_name'], how='left')

# 4) Mark breaches: value > threshold_p95 (treat missing threshold as non-breach)
log_win['above_threshold'] = log_win['value'] > log_win['threshold_p95']

# 5) Aggregate per (cmdb_id, log_name)
def earliest_breach(series_ts, series_flag):
    # return min timestamp where flag True, else NaT
    if series_flag.any():
        return series_ts[series_flag].min()
    else:
        return pd.NaT

grouped = (
    log_win
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .apply(lambda df: pd.Series({
        'threshold_p95': df['threshold_p95'].iloc[0],
        'points_in_window': df.shape[0],
        'points_above_threshold': int(df['above_threshold'].sum()),
        'earliest_breach_time_utc': earliest_breach(df['timestamp'], df['above_threshold']),
        'max_value_in_window': df['value'].max()
    }))
    .reset_index()
)

# 6) Compute ratio_max_to_threshold (handle non-positive thresholds)
grouped['ratio_max_to_threshold'] = np.where(
    grouped['threshold_p95'] > 0,
    grouped['max_value_in_window'] / grouped['threshold_p95'],
    np.nan
)

# 7) Sort and limit to top 20 rows as requested
final_table = grouped.sort_values(
    by=['points_above_threshold', 'ratio_max_to_threshold'],
    ascending=[False, False]
).head(20).reset_index(drop=True)

# 8) Count how many (cmdb_id, log_name) pairs had any breaches in the window
breach_count = int((grouped['points_above_threshold'] > 0).sum())

# Keep output compact: selected columns and small table
final_table = final_table[
    ['cmdb_id','log_name','threshold_p95','points_in_window','points_above_threshold',
     'earliest_breach_time_utc','max_value_in_window','ratio_max_to_threshold']
]

final_table, breach_count

```
Out[4]:
```
Summary of log analysis (window 2024-01-20 06:09:06–06:39:06 UTC, components carts/catalogue/orders/payment/user):

- Total (cmdb_id, log_name) pairs analyzed: 10 (5 components × 2 log names).
- Pairs with any breaches (value > global P95): 4.

Breaches (compact):
1. user / log.row_count — P95 = 614.8; points_in_window = 25; points_above_threshold = 2; earliest breach = 2024-01-20 06:15:00 UTC; max = 622; ratio ≈ 1.0117
2. orders / log.row_count — P95 = 128.0; points_in_window = 25; points_above_threshold = 1; earliest breach = 2024-01-20 06:15:00 UTC; max = 134; ratio ≈ 1.0469
3. payment / log.row_count — P95 = 104.0; points_in_window = 25; points_above_threshold = 1; earliest breach = 2024-01-20 06:15:00 UTC; max = 107; ratio ≈ 1.0288
4. catalogue / log.row_count — P95 = 166.0; points_in_window = 25; points_above_threshold = 1; earliest breach = 2024-01-20 06:16:00 UTC; max = 167; ratio ≈ 1.0060

Notes:
- All breaches are modest (ratios ≈ 1.01–1.05).
- log.error_count had no breaches for any of the five components in the window.
- carts / log.row_count had no breaches (points_in_window = 24, max = 100, equals P95).

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id         log_name  threshold_p95  points_in_window  points_above_threshold  earliest_breach_time_utc  max_value_in_window  ratio_max_to_threshold
0       user    log.row_count          614.8                25                       2 2024-01-20 06:15:00+00:00                  622                1.011711
1     orders    log.row_count          128.0                25                       1 2024-01-20 06:15:00+00:00                  134                1.046875
2    payment    log.row_count          104.0                25                       1 2024-01-20 06:15:00+00:00                  107                1.028846
3  catalogue    log.row_count          166.0                25                       1 2024-01-20 06:16:00+00:00                  167                1.006024
4      carts    log.row_count          100.0                24                       0                       NaT                  100                1.000000
5      carts  log.error_count            0.0                24                       0                       NaT                    0                     NaN
6  catalogue  log.error_count            0.0                25                       0                       NaT                    0                     NaN
7     orders  log.error_count            0.0                25                       0                       NaT                    0                     NaN
8    payment  log.error_count            0.0                25                       0                       NaT                    0                     NaN
9       user  log.error_count            0.0                25                       0                       NaT                    0                     NaN, 4)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse existing metric DataFrame; ensure timestamp is datetime UTC
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Components of interest
components = ["carts","catalogue","orders","payment","user"]

# 1) Compute global P95 thresholds per (cmdb_id, kpi_name) using the full series
thresholds = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='threshold_p95')
)

# 2) Filter metric rows to the incident window and restrict to target components
start = pd.to_datetime('2024-01-20 06:09:06', utc=True)
end = pd.to_datetime('2024-01-20 06:39:06', utc=True)
mw = metric[
    (metric['timestamp'] >= start) &
    (metric['timestamp'] <= end) &
    (metric['cmdb_id'].isin(components))
].copy()

# 3) Merge thresholds so each row has its group's P95
mw = mw.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Mark anomalies: value > threshold_p95 (treat missing threshold as non-anomalous)
mw['anomaly'] = mw['value'] > mw['threshold_p95']

# 5) Keep only anomalous rows for fault grouping
anoms = mw[mw['anomaly']].sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)

# Prepare an empty faults DataFrame in case there are no anomalies
faults_df = pd.DataFrame(columns=[
    'cmdb_id','kpi_name','threshold_p95','fault_start_utc','fault_end_utc',
    'points_in_fault','max_value_in_fault','ratio_max_to_threshold'
])

if not anoms.empty:
    # 6) Compute gaps within each (cmdb_id,kpi_name) to split faults when gap > 60s
    anoms['prev_ts'] = anoms.groupby(['cmdb_id','kpi_name'])['timestamp'].shift(1)
    anoms['gap_s'] = (anoms['timestamp'] - anoms['prev_ts']).dt.total_seconds().fillna(0)
    anoms['new_fault'] = (anoms['gap_s'] > 60).astype(int)
    anoms['fault_seq'] = anoms.groupby(['cmdb_id','kpi_name'])['new_fault'].cumsum()

    # 7) Aggregate per fault
    grouped = anoms.groupby(['cmdb_id','kpi_name','fault_seq'], dropna=False).agg(
        threshold_p95=('threshold_p95', 'first'),
        fault_start_utc=('timestamp', 'min'),
        fault_end_utc=('timestamp', 'max'),
        points_in_fault=('timestamp', 'size'),
        max_value_in_fault=('value', 'max')
    ).reset_index(drop=False)

    # 8) Compute ratio (handle zero/non-positive thresholds)
    grouped['ratio_max_to_threshold'] = np.where(
        grouped['threshold_p95'] > 0,
        grouped['max_value_in_fault'] / grouped['threshold_p95'],
        np.nan
    )

    faults_df = grouped[[
        'cmdb_id','kpi_name','threshold_p95','fault_start_utc','fault_end_utc',
        'points_in_fault','max_value_in_fault','ratio_max_to_threshold'
    ]].copy()

# 9) Per-component summary: one row per component (exactly the five components)
# For components with no faults, fill defaults
comp_rows = []
for comp in components:
    comp_faults = faults_df[faults_df['cmdb_id'] == comp]
    total_faults = int(comp_faults.shape[0])
    if total_faults == 0:
        comp_rows.append({
            'cmdb_id': comp,
            'total_faults': 0,
            'earliest_fault_start_utc': pd.NaT,
            'max_ratio_over_threshold_among_faults': np.nan,
            'most_severe_kpi': None,
            'time_of_most_severe_fault_start_utc': pd.NaT
        })
    else:
        # find fault with max ratio
        idx_max = comp_faults['ratio_max_to_threshold'].idxmax()
        max_ratio = float(comp_faults.loc[idx_max, 'ratio_max_to_threshold'])
        severe_kpi = comp_faults.loc[idx_max, 'kpi_name']
        severe_start = comp_faults.loc[idx_max, 'fault_start_utc']
        earliest_start = comp_faults['fault_start_utc'].min()
        comp_rows.append({
            'cmdb_id': comp,
            'total_faults': total_faults,
            'earliest_fault_start_utc': earliest_start,
            'max_ratio_over_threshold_among_faults': max_ratio,
            'most_severe_kpi': severe_kpi,
            'time_of_most_severe_fault_start_utc': severe_start
        })

comp_summary = pd.DataFrame(comp_rows)

# 10) Count how many of the five components have max_ratio >= 1.2
count_of_components_with_max_ratio_ge_1_2 = int((comp_summary['max_ratio_over_threshold_among_faults'] >= 1.2).sum())

# Final compact outputs
comp_summary, count_of_components_with_max_ratio_ge_1_2

```
Out[5]:
```
summary = (
    "Per-component fault summary (window 2024-01-20 06:09:06–06:39:06 UTC):\n\n"
    "1) carts: total_faults = 11; earliest_fault_start_utc = 2024-01-20 06:12:00+00:00; "
    "max_ratio_over_threshold_among_faults = 1.038511; most_severe_kpi = cpu; "
    "time_of_most_severe_fault_start_utc = 2024-01-20 06:28:00+00:00\n\n"
    "2) catalogue: total_faults = 9; earliest_fault_start_utc = 2024-01-20 06:12:00+00:00; "
    "max_ratio_over_threshold_among_faults = 1.007664; most_severe_kpi = workload; "
    "time_of_most_severe_fault_start_utc = 2024-01-20 06:27:00+00:00\n\n"
    "3) orders: total_faults = 12; earliest_fault_start_utc = 2024-01-20 06:19:00+00:00; "
    "max_ratio_over_threshold_among_faults = 1.036217; most_severe_kpi = workload; "
    "time_of_most_severe_fault_start_utc = 2024-01-20 06:36:00+00:00\n\n"
    "4) payment: total_faults = 9; earliest_fault_start_utc = 2024-01-20 06:15:00+00:00; "
    "max_ratio_over_threshold_among_faults = 1.066649; most_severe_kpi = cpu; "
    "time_of_most_severe_fault_start_utc = 2024-01-20 06:32:00+00:00\n\n"
    "5) user: total_faults = 9; earliest_fault_start_utc = 2024-01-20 06:15:00+00:00; "
    "max_ratio_over_threshold_among_faults = 1.035066; most_severe_kpi = latency-90; "
    "time_of_most_severe_fault_start_utc = 2024-01-20 06:35:00+00:00\n\n"
    "Note: count_of_components_with_max_ratio_ge_1_2 = 0"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  total_faults  earliest_fault_start_utc  max_ratio_over_threshold_among_faults most_severe_kpi time_of_most_severe_fault_start_utc
0      carts            11 2024-01-20 06:12:00+00:00                               1.038511             cpu           2024-01-20 06:28:00+00:00
1  catalogue             9 2024-01-20 06:12:00+00:00                               1.007664        workload           2024-01-20 06:27:00+00:00
2     orders            12 2024-01-20 06:19:00+00:00                               1.036217        workload           2024-01-20 06:36:00+00:00
3    payment             9 2024-01-20 06:15:00+00:00                               1.066649             cpu           2024-01-20 06:32:00+00:00
4       user             9 2024-01-20 06:15:00+00:00                               1.035066      latency-90           2024-01-20 06:35:00+00:00, 0)```
```